# Adding detector-variation systematics (DetVar)

Covers adding detector-variation (`'detv'`) systematics on top of
step3's rate/norm/MCstat ones, via `funcs.py`'s `systs_input()`, plus a
final conformance check before shipping a new analysis.

| Section | File | What you set |
|---|---|---|
| 1 | -- | load MC + data (same pattern as step3's section 2) |
| 2 | `detvar.py` | register your analysis (`SLC_KEY`, `DEFAULT_PREPROCESS_FN`, `SELECTION_FN_MAP`) |
| 3 | `config.py` | make `detvars.h5` + set `DETVAR_DIR`/`DETVAR_DICT_FILES` |
| 4 | `funcs.py` | add `'detv'` to `systs_input()` |
| 5 | `plotting.py` | plot the systematics category breakdown |
| 6 | -- | conformance check |

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# TODO: set to the PARENT directory of your cafpybara/ clone -- NOT cafpybara/ itself (skipped if already importable).
sys.path.append("TODO_SET_ME/parent_dir_of_cafpybara")

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

In [ ]:
#MC_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/nu2026_df/mcbnb_cv/mcbnb_wgt_full_v4.df"
MC_FILE = "/path/to/your_mc_file.h5"
#DATA_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/nu2026_df/dtbnb/fixed_bnblight_2026a_v4.df"
DATA_FILE = "/path/to/your_data_file.h5"
#OFFBEAM_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/nu2026_df/dtoff/offbeamlight_v4.df"
OFFBEAM_FILE = "/path/to/your_offbeam_file.h5"
#VAR = "your_variable"
#BINS = [0, 1, 2, 3]
VAR = ('primshw', 'shw', 'bestplane_energy', '', '', '') #TEST
BINS = np.linspace(0, 1, 10) #TEST

Same single load step as step3's section 2, reused here for the detvar
demo.

* **TODO:**
  * pass `preprocess_fn=getattr(ana, 'preprocess_databnb', ana.preprocess_data)`
    to the on-beam call, and the off-beam equivalent to the off-beam call --
    see step2a A3/A4; falls back to the no-op `preprocess_data` if unset.
  * double-check `f` is the most up-to-date off-beam scaling fudge factor
    (see step2a A4).

## 1. Load MC + data

Reuses the same weighted `MC_FILE`/`DATA_FILE`/`OFFBEAM_FILE` from
step3 -- see step3's section 1 for how to produce them with `-ngrid`/
`concat_hdf.py`.

* **TODO:** set `MC_FILE`, `DATA_FILE`, `OFFBEAM_FILE`

In [ ]:
try:
    mc_df, mc_pot, mc_ngen = ana.load_mc(MC_FILE, cuts=ana.DEFAULT_CUTS)

    data_df, data_pot, data_ngates = ana.load_data(
        DATA_FILE, cuts=ana.DEFAULT_CUTS,
        preprocess_fn=getattr(ana, 'preprocess_databnb', ana.preprocess_data),
    )

    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False,
        offbeam_signal_value=ana.signal_dict["offbeam"],
        cuts=ana.DEFAULT_CUTS,
        preprocess_fn=getattr(ana, 'preprocess_dataoff', ana.preprocess_data),
    )

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor (see step2a A4)
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates

    weight_col = ('weights_mc', '', '', '', '', '')
    mc_df[weight_col] = 1.0
    offbeam_df[weight_col] = scale_offbeam

    mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample']
    mc_top_level = set(mc_df.columns.get_level_values(0)) | set(mc_df.index.names)
    if 'file_idx' in mc_top_level:
        mcstat_cols.append('file_idx')  # only from concat_hdf.py (see step3's section 1) -- fine if absent here

    mc_df = pd.concat([
        ana.mcstat(mc_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
        offbeam_df.assign(sample=1).set_index("sample", append=True).reset_index(),
    ])
    print("Merged MC + off-beam rows:", len(mc_df))

    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot)  # no systs= yet
except Exception as e:
    mc_df, data_df = None, None
    print("No real MC/data/offbeam file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## 2. Register your analysis in `detvar.py`

`core/detvar/process_detvars.py` dispatches per-analysis via
`analyses/<name>/detvar.py` -- a new analysis needs one.

* **TODO:** create `/my_analysis/detvar.py` with `SLC_KEY`,
  `DEFAULT_PREPROCESS_FN`, and `SELECTION_FN_MAP` (selection-type name ->
  `(cuts, stage)`)

Example (`analyses/hnlpi0/detvar.py`):
```python
from . import preprocess
from . import analysis

SLC_KEY = "rec"

DEFAULT_PREPROCESS_FN = preprocess.preprocess_mcbnb

SELECTION_FN_MAP = {
    "1shw": (analysis.PI0_CUT_LISTS['1shw'], None),
}
```

> ⚠️ a `"preprocess"` entry (no cuts, no selection) is injected into `SELECTION_FN_MAP` automatically -- don't add it yourself.

See below for the sanity check:

In [ ]:
print("SLC_KEY:", ana.detvar.SLC_KEY)
print("SELECTION_FN_MAP keys:", list(ana.detvar.SELECTION_FN_MAP.keys()))
try:
    ana.detvar.DEFAULT_PREPROCESS_FN(None)
except Exception as e:
    print("Expected -- DEFAULT_PREPROCESS_FN not set yet:")
    print(f"  {type(e).__name__}: {e}")

## 3. Make `detvars.h5` and set `config.py`

* **TODO:**
  1. Produce raw `detvar_*.df` files:
     ```bash
     python run_df_maker.py -c ./analysis_village/hnl_nuee_nupi0/configs/hnl_mcnu_detvar.py -l <input.list> -o <output_name>
     ```
  2. Turn those into `detvars.h5`:
     ```bash
     python core/detvar/process_detvars.py --analysis my_analysis -i <raw_detvar_dir> -o <output_dir> -s preprocess
     ```
  3. Set `DETVAR_DIR`/`DETVAR_DICT_FILES` in `/my_analysis/config.py` to
     point at that store.
* **TODO (optional):** for a real in-time-cosmic sample, set `config.py`'s
  `INTIME_FILE` and add `'cosmic'` to `systs_input()`'s `uncertainty_keys`
  -- left unset, requesting `'cosmic'` raises instead of silently skipping it.

Example (`analyses/hnlpi0/config.py`):
```python
HNL_DETVAR_DIR = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/July2026/detvar"
HNL_DETVAR_DICT_FILES = [HNL_DETVAR_DIR + "/detvars.h5",]
```

See below for the sanity check:

In [ ]:
print("DETVAR_DICT_FILES:", ana.config.DETVAR_DICT_FILES)

## 4. Add `'detv'` to `systs_input()` in `funcs.py`

Add `'detv'` to `uncertainty_keys` to fold detector-systematic
uncertainty into the covariance matrix. `systs_input()` defaults
`detvar_files` to `config.py`'s `DETVAR_DICT_FILES` from section 3 if
you don't pass `detvar_dict=`/`detvar_files=` explicitly. Under the
hood, `get_total_cov` re-applies the same `cuts=` used to build `mc_df`
to the CV/DV samples via `core.syst.get_detvar_systs`.

* **TODO:** none new -- reuses `DEFAULT_CUTS`/`DETVAR_DICT_FILES` already set.

> ⚠️ omitting `cuts=` when `'detv'`/`'cosmic'` is in `uncertainty_keys` raises a `ValueError` instead of silently falling back to an unselected comparison.

See below for the actual call:

In [ ]:
# What you'd actually write in your own analysis notebook:
if mc_df is not None:
    si = ana.systs_input(mc_pot, uncertainty_keys={'rate', 'norm', 'detv'})
    print("leaving detvar_dict/detvar_files unset falls back to config.py:")
    print("si.detvar_files is ana.config.DETVAR_DICT_FILES:", si.detvar_files is ana.config.DETVAR_DICT_FILES)

    try:
        detvar_syst_output = ana.get_total_cov(mc_df, reco_var=VAR, bins=BINS, **si.to_kwargs())
        print("detvar_syst_output.rate_cov shape:", detvar_syst_output.rate_cov.shape)
    except Exception as e:
        detvar_syst_output = None
        print("DETVAR_DICT_FILES is still config.py's placeholder path -- expected:")
        print(f"  {type(e).__name__}: {e}")
else:
    detvar_syst_output = None
    print("No real MC file here (this notebook needs none) -- see section 1 above")

In [ ]:
# plot_mc_data with systematics on (reuses detvar_syst_output from above):
if mc_df is not None and data_df is not None and detvar_syst_output is not None:
    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot, systs=detvar_syst_output)
else:
    print("No real data file here (this notebook needs none) -- see section 4 above")

## 5. Plot the systematics category breakdown

* **TODO (optional):** inspect `plot_syst_category_breakdown`'s docstring
  for plotting options (figure size, projected POT, xsec mode, etc.)

In [ ]:
if detvar_syst_output is not None:
    fig, axes, cats, cat_sums = ana.plot_syst_category_breakdown(
        syst_vars=[(detvar_syst_output, BINS, VAR)],
        category_dict=ana.category_dict_signal,
    )
    plt.show()
else:
    print("No real MC file here (this notebook needs none) -- see section 4 above")

## 6. Before you ship it: a manual conformance pass

* **TODO (optional):** run `scripts/verify_new_analysis.py <name>` --
  automates this check, plus two this notebook can't: a
  no-op-preprocessing-default audit, and an `ast` sweep of example
  notebooks' `systs_input`/`SystematicsInput`/`PlottingConfig` calls for
  dead kwargs. If your loader needs real preprocessing, add it to that
  script's `NOOP_DEFAULT_CHECKS` table (curated per-analysis).

See below for the same idea, inline:

In [ ]:
def conformance_check(analysis_module):
    """Run this against your real analysis module, not the template."""
    problems = []

    for mod_name in ['io', 'funcs', 'plotting', 'selection', 'physics', 'syst', 'classes', 'preprocess', 'detvar']:
        core_mod = getattr(core, mod_name)
        missing = [n for n in getattr(core_mod, '__all__', []) if not hasattr(analysis_module, n)]
        if missing:
            problems.append(f"core.{mod_name} not fully re-exported: missing {missing}")

    if analysis_module.get_total_cov is not core.funcs.get_total_cov:
        problems.append("get_total_cov has been overridden -- delete it, keep only systs_input()")

    if not hasattr(analysis_module, 'systs_input'):
        problems.append("no systs_input() factory found")

    if problems:
        print(f"{len(problems)} problem(s) found:")
        for p in problems:
            print(f"  - {p}")
    else:
        print("No problems found by this (non-exhaustive) check.")
    return problems


conformance_check(ana)